In [3]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import src.config as config
from src.dataset import get_embeddings, TestDataset, PrecomputedDataset, mashup_collate_fn, test_collate_fn
from src.models import EffNetB0
from src.utils import get_train_stats
from src.metrics import compute_fid

In [4]:
mean, std = get_train_stats(config.TRAIN_STATS_PATH)
test_files = sorted([f for f in Path(config.TEST_DIR).iterdir() if f.is_file()])

test_set = TestDataset(test_files, mean=mean, std=std)
train_set = PrecomputedDataset("precomputed/train_v4_list.txt", mean=mean, std=std)

effnet_finetuned = EffNetB0(num_classes=10, pretrained=False, freeze_backbone=True)
ckpt = torch.load(Path(config.CHECKPOINTS_DIR) / "effnetb0.ckpt", map_location="cpu", weights_only=False)
effnet_finetuned.load_state_dict(ckpt["model_state"])

<All keys matched successfully>

In [ ]:
test_loader = DataLoader(
    dataset=test_set,
    batch_size=16,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    collate_fn=test_collate_fn
)

train_loader = DataLoader(
    dataset=train_set,
    batch_size=16,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    collate_fn=mashup_collate_fn,
)

test_embeddings = get_embeddings(model=effnet_finetuned, loader=test_loader)
train_embeddings = get_embeddings(model=effnet_finetuned, loader=train_loader)

print(test_embeddings.shape, train_embeddings.shape)
print(compute_fid(train_embeddings, test_embeddings))